# 02 — TCGA Data Loading

This notebook loads and merges the two TCGA pan-cancer datasets used for TP53 mutation prediction.

> **Files must already be placed in `data/raw/`** before running. See the project README for download instructions.

---

## Data sources

| File | Description |
|------|-------------|
| `EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2-v2.geneExp.tsv` | TCGA PANCAN RNA-seq expression matrix — EB++ batch-corrected RSEM values; genes × samples (1.8 GB) |
| `mc3.v0.2.8.PUBLIC.maf` | MC3 pan-cancer somatic mutation calls in MAF format; one row per mutation event (~2.5 GB uncompressed) |

---

## What this notebook produces

A single merged DataFrame — **samples (patients) × genes**, with two label columns appended:

| Column | Type | Description |
|--------|------|-------------|
| `tp53_binary` | int (0/1) | 1 if the patient has a PASS-quality TP53 mutation |
| `Variant_Classification` | str | Mutation type (e.g. `Missense_Mutation`) or `WT` for wild-type |

Output saved to: `data/processed/tcga_merged_raw.csv.gz`

---

## Why TCGA is harder than CCLE

| Aspect | CCLE | TCGA |
|--------|------|------|
| Sample type | Cancer cell lines (homogeneous) | Primary tumors (heterogeneous) |
| Batch effects | Minimal (one lab) | Substantial (multi-center) — mitigated here by EB++ correction |
| Mutation calling | Single pipeline | MC3 uses multi-caller consensus (more conservative) |
| Expression scale | log₂(TPM+1) pre-normalised | Raw RSEM — must log-transform during preprocessing |
| Class noise | Low | Higher — tumor purity, stromal contamination |

## Setup

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

EXPR_FILE = RAW_DIR / "EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2-v2.geneExp.tsv"
MAF_FILE  = RAW_DIR / "mc3.v0.2.8.PUBLIC.maf"

for f in (EXPR_FILE, MAF_FILE):
    status = "OK" if f.exists() else "MISSING"
    print(f"[{status}] {f.name}")

[OK] EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2-v2.geneExp.tsv
[OK] mc3.v0.2.8.PUBLIC.maf


---

## 1. Gene expression

### 1a. Load the raw TSV

The file is 1.8 GB and has ~20,500 genes (rows) × ~11,000 samples (columns). Expect ~2 minutes to load.

In [8]:
print("Loading gene expression (~1.8 GB) — this takes ~2 min...")
expr_raw = pd.read_csv(EXPR_FILE, sep='\t', index_col=0)
print(f"Raw shape: {expr_raw.shape}  (genes × samples)")
print(f"First 5 gene IDs : {expr_raw.index[:5].tolist()}")
print(f"First 3 barcodes : {expr_raw.columns[:3].tolist()}")

Loading gene expression (~1.8 GB) — this takes ~2 min...
Raw shape: (20531, 11069)  (genes × samples)
First 5 gene IDs : ['?|100130426', '?|100133144', '?|100134869', '?|10357', '?|10431']
First 3 barcodes : ['TCGA-OR-A5J1-01A-11R-A29S-07', 'TCGA-OR-A5J2-01A-11R-A29S-07', 'TCGA-OR-A5J3-01A-11R-A29S-07']


### 1b. Parse gene symbols

Gene IDs are stored as `SYMBOL|ENTREZ_ID` (e.g., `TP53|7157`). Rows with symbol `?` have no gene annotation and are discarded.

In [9]:
# Strip Entrez ID suffix, keep symbol only
expr_raw.index = expr_raw.index.str.split('|').str[0]

# Drop genes with no symbol
n_before = len(expr_raw)
expr_raw = expr_raw.loc[expr_raw.index != '?']
print(f"Removed {n_before - len(expr_raw):,} genes with '?' symbol")

# Drop duplicate gene symbols (keep first occurrence — rare but can happen for multi-transcript genes)
n_dupes = expr_raw.index.duplicated().sum()
if n_dupes > 0:
    print(f"Removing {n_dupes:,} duplicate gene entries (keeping first)")
    expr_raw = expr_raw[~expr_raw.index.duplicated(keep='first')]

print(f"Genes retained: {len(expr_raw):,}")

Removed 29 genes with '?' symbol
Removing 1 duplicate gene entries (keeping first)
Genes retained: 20,501


### 1c. Transpose and filter to primary tumor samples

After transposing, each row is a sample. We keep only **primary solid tumors** (sample-type code `01`) and **primary blood-derived cancers** (code `03`). Normal tissue samples (code `11`) are excluded because we are predicting tumor-derived TP53 mutations.

TCGA barcode structure:
```
TCGA - {TSS} - {participant} - {sampletype}{vial} - ...
   [0:4]  [5:6]    [8:11]         [13:14]    [15]
```
The 12-character patient ID is `barcode[:12]`.

In [10]:
# Transpose: genes as columns, samples as rows
expr = expr_raw.T.copy()
expr.index.name = 'sample_barcode'
print(f"After transpose: {expr.shape}  (samples × genes)")

# Sample type is characters 13-14 of the full TCGA barcode
sample_types = expr.index.str[13:15]
print("\nSample type distribution:")
print(pd.Series(sample_types).value_counts().to_string())

# Keep primary tumors only
primary_mask = sample_types.isin(['01', '03'])
expr = expr.loc[primary_mask]
print(f"\nAfter primary-tumor filter: {expr.shape[0]:,} samples")

After transpose: (11069, 20501)  (samples × genes)

Sample type distribution:
sample_barcode
01    9706
11     737
06     395
03     173
02      46
05      11
07       1

After primary-tumor filter: 9,879 samples


In [11]:
# Standardise to 12-char patient barcode: TCGA-{TSS}-{participant}
expr.index = expr.index.str[:12]
expr.index.name = 'patient_id'

# If a patient appears more than once (multiple tumor samples), keep the first
n_dupes = expr.index.duplicated().sum()
if n_dupes > 0:
    print(f"Removing {n_dupes:,} duplicate patient entries (keeping first sample)")
    expr = expr[~expr.index.duplicated(keep='first')]

print(f"Unique patients in expression data: {expr.shape[0]:,}")

Removing 4 duplicate patient entries (keeping first sample)


Unique patients in expression data: 9,875


---

## 2. MC3 somatic mutations

### 2a. Load MAF

The MC3 file covers all TCGA somatic mutations across cancer types. We only need a handful of columns, so we use `usecols` to avoid loading the full 114-column file into memory.

In [12]:
MAF_COLS = [
    'Hugo_Symbol', 'Tumor_Sample_Barcode',
    'Variant_Classification', 'Variant_Type',
    'HGVSp_Short', 'SIFT', 'PolyPhen', 'FILTER'
]

print("Loading MC3 MAF (~2.5 GB uncompressed) — this takes ~1 min...")
maf = pd.read_csv(MAF_FILE, sep='\t', usecols=MAF_COLS, low_memory=False)
print(f"MAF shape: {maf.shape}")
print(f"Unique samples : {maf['Tumor_Sample_Barcode'].nunique():,}")
print(f"Unique patients: {maf['Tumor_Sample_Barcode'].str[:12].nunique():,}")

Loading MC3 MAF (~2.5 GB uncompressed) — this takes ~1 min...
MAF shape: (3600963, 8)
Unique samples : 10,295
Unique patients: 10,224


### 2b. Filter for TP53 PASS-quality mutations

In [13]:
# Keep only TP53 mutations that passed all MC3 quality filters
tp53 = maf[(maf['Hugo_Symbol'] == 'TP53') & (maf['FILTER'] == 'PASS')].copy()

print(f"TP53 PASS mutations : {len(tp53):,}")
print(f"Unique patients      : {tp53['Tumor_Sample_Barcode'].str[:12].nunique():,}")
print("\nVariant_Classification breakdown:")
print(tp53['Variant_Classification'].value_counts().to_string())

TP53 PASS mutations : 3,729
Unique patients      : 3,325

Variant_Classification breakdown:
Variant_Classification
Missense_Mutation         2382
Nonsense_Mutation          488
Frame_Shift_Del            332
Splice_Site                238
Frame_Shift_Ins            101
Silent                      90
In_Frame_Del                68
In_Frame_Ins                11
Intron                       8
3'UTR                        7
5'UTR                        3
Translation_Start_Site       1


### 2c. Resolve multiple TP53 mutations per patient

Some patients have more than one somatic TP53 mutation (compound heterozygous or multiple hits). We retain the most functionally severe mutation using a predefined priority order.

| Priority | Classification |
|----------|----------------|
| 0 (highest) | Nonsense_Mutation |
| 1 | Frame_Shift_Del |
| 2 | Frame_Shift_Ins |
| 3 | Splice_Site |
| 4 | Missense_Mutation |
| 5 | In_Frame_Del |
| 6 | In_Frame_Ins |
| 7+ | Translation_Start_Site, Nonstop, Silent, … |

In [14]:
SEVERITY = {
    'Nonsense_Mutation':         0,
    'Frame_Shift_Del':           1,
    'Frame_Shift_Ins':           2,
    'Splice_Site':               3,
    'Missense_Mutation':         4,
    'In_Frame_Del':              5,
    'In_Frame_Ins':              6,
    'Translation_Start_Site':    7,
    'Nonstop_Mutation':          8,
    'Silent':                    9,
}

tp53['patient_id'] = tp53['Tumor_Sample_Barcode'].str[:12]
tp53['_sev'] = tp53['Variant_Classification'].map(SEVERITY).fillna(99)

n_before = tp53['patient_id'].nunique()
tp53 = (
    tp53
    .sort_values('_sev')
    .drop_duplicates(subset='patient_id', keep='first')
    .drop(columns=['_sev', 'Tumor_Sample_Barcode'])
    .set_index('patient_id')
)
print(f"Patients before dedup: {n_before:,}")
print(f"Patients after dedup : {len(tp53):,}")
print("\nFinal Variant_Classification breakdown:")
print(tp53['Variant_Classification'].value_counts().to_string())

Patients before dedup: 3,325
Patients after dedup : 3,325

Final Variant_Classification breakdown:
Variant_Classification
Missense_Mutation    2069
Nonsense_Mutation     482
Frame_Shift_Del       319
Splice_Site           230
Frame_Shift_Ins        99
In_Frame_Del           62
Silent                 45
In_Frame_Ins            9
Intron                  5
3'UTR                   4
5'UTR                   1


---

## 3. Merge expression + TP53 labels

Left join: all expression patients are kept. Patients absent from the mutation table are labelled wild-type.

In [15]:
merged = expr.copy()

# Binary label: 1 = TP53-mutant, 0 = wild-type
merged['tp53_binary'] = merged.index.isin(tp53.index).astype(int)

# Multi-class label: mutation type or 'WT'
merged['Variant_Classification'] = (
    merged.index.map(tp53['Variant_Classification']).fillna('WT')
)

print(f"Merged shape: {merged.shape}")
print(f"  Gene columns : {merged.shape[1] - 2:,}")
print(f"  Label columns: 2 (tp53_binary, Variant_Classification)")
print()
print("tp53_binary counts:")
print(merged['tp53_binary'].value_counts().rename({0: 'WT (0)', 1: 'Mutated (1)'}).to_string())
print()
print("Variant_Classification counts:")
print(merged['Variant_Classification'].value_counts().to_string())

Merged shape: (9875, 20503)
  Gene columns : 20,501
  Label columns: 2 (Mutated, Variant_Classification)

Mutated counts:
Mutated
WT (0)         6744
Mutated (1)    3131

Variant_Classification counts:
Variant_Classification
WT                   6744
Missense_Mutation    1949
Nonsense_Mutation     452
Frame_Shift_Del       298
Splice_Site           215
Frame_Shift_Ins        96
In_Frame_Del           59
Silent                 43
In_Frame_Ins            9
Intron                  5
3'UTR                   4
5'UTR                   1


---

## 4. Quick sanity checks

In [16]:
# Check TP53 gene is present in expression data
tp53_expr_col = 'TP53' if 'TP53' in merged.columns else None
if tp53_expr_col:
    print("TP53 expression column found.")
    print(merged['TP53'].describe())
else:
    print("WARNING: TP53 column not found — check gene parsing step.")

TP53 expression column found.
count     9875.000000
mean      1638.405671
std       1082.900491
min         26.399700
25%        914.115000
50%       1399.270000
75%       2150.925000
max      15568.471849
Name: TP53, dtype: float64


In [17]:
# Check for completely empty rows (failed samples)
zero_row_mask = (merged.iloc[:, :-2] == 0).all(axis=1)
print(f"Samples with all-zero expression: {zero_row_mask.sum():,}")
if zero_row_mask.sum() > 0:
    print("  These rows will be investigated further in the EDA notebook.")

Samples with all-zero expression: 0


In [18]:
# Check for any NaN values in expression columns
nan_count = merged.iloc[:, :-2].isnull().sum().sum()
print(f"NaN values in expression matrix: {nan_count:,}")

NaN values in expression matrix: 4,069,730


---

## 5. Save

Save as gzip-compressed CSV. This file is the input for both the EDA and preprocessing notebooks.

In [19]:
OUT_FILE = PROCESSED_DIR / "tcga_merged_raw.csv.gz"
print(f"Saving to {OUT_FILE} ...")
merged.to_csv(OUT_FILE, compression='gzip')
print(f"Done. File size: {OUT_FILE.stat().st_size / 1e6:.1f} MB")

Saving to ../data/processed/tcga_merged_raw.csv.gz ...
Done. File size: 697.4 MB


---

## Summary

| | Value |
|---|---|
| Expression patients | see output above |
| Genes | see output above |
| TP53-mutant patients | see `tp53_binary` counts |
| Wild-type patients | see `tp53_binary` counts |
| Output file | `data/processed/tcga_merged_raw.csv.gz` |

Next: `03_tcga_eda.ipynb` for exploratory analysis before cleaning.